# 01 — Single-Layer Perceptron From Scratch

**Objective:** Implement a single-layer perceptron using Python and NumPy *only* (no ML frameworks).
We will understand every component — weights, bias, activation, and the perceptron learning rule — and
train the model on a linearly separable dataset. We will also demonstrate why XOR cannot be learned by a single perceptron.

---
## 1. What is a Perceptron?

A **perceptron** is the simplest form of a neural network — a single computational unit that:

1. Takes **n input features** `x1, x2, ..., xn`.
2. Multiplies each by a **weight** `w1, w2, ..., wn` and adds a **bias** `b`.
3. Computes a **weighted sum** (net input): `z = w1*x1 + w2*x2 + ... + wn*xn + b`.
4. Passes `z` through an **activation function** to produce a prediction.
5. Compares the prediction to the true label and updates weights using the **perceptron learning rule**.

### Binary Step Activation

```
f(z) = 1  if z >= 0
       0  otherwise
```

### Perceptron Learning Rule

```
w_i <- w_i + lr * (y - y_hat) * x_i
b   <- b   + lr * (y - y_hat)
```

where `lr` is the learning rate, `y` is the true label, and `y_hat` is the predicted label.

---
## 2. Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Reproducibility
np.random.seed(42)

# --- Hyperparameters (adjust freely) ---
LEARNING_RATE = 0.1
EPOCHS        = 20
# ----------------------------------------

print(f"NumPy version : {np.__version__}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Epochs        : {EPOCHS}")

---
## 3. Datasets

We use the AND and OR logical functions — both are **linearly separable**, meaning a single straight line (decision boundary) can separate the two classes.

In [ ]:
# AND truth table
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1], dtype=float)

# OR truth table
X_or  = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_or  = np.array([0, 1, 1, 1], dtype=float)

# XOR truth table  (NOT linearly separable)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

print("AND dataset:")
for x, y in zip(X_and, y_and):
    print(f"  {x} -> {int(y)}")

print("\nOR dataset:")
for x, y in zip(X_or, y_or):
    print(f"  {x} -> {int(y)}")

print("\nXOR dataset:")
for x, y in zip(X_xor, y_xor):
    print(f"  {x} -> {int(y)}")

---
## 4. Perceptron Implementation (NumPy only)

In [ ]:
class Perceptron:
    """Single-layer perceptron with binary step activation.
    Implemented from scratch using NumPy only.
    """

    def __init__(self, n_features, learning_rate=0.1):
        """Initialize weights and bias to small random values."""
        self.lr = learning_rate
        # Small random weights; bias starts at zero
        self.weights = np.random.uniform(-0.5, 0.5, size=n_features)
        self.bias    = 0.0
        self.errors_per_epoch = []

    @staticmethod
    def _step(z):
        """Binary step activation: return 1 if z >= 0, else 0."""
        return np.where(z >= 0, 1, 0).astype(float)

    def _net_input(self, X):
        """Compute weighted sum: z = X . w + b."""
        return X @ self.weights + self.bias

    def predict(self, X):
        """Predict binary class labels (0 or 1) for each sample."""
        return self._step(self._net_input(X))

    def fit(self, X, y, epochs=10):
        """Train using the perceptron learning rule.
        Weight update: w += lr * (y - y_hat) * x
        Bias   update: b += lr * (y - y_hat)
        """
        for epoch in range(1, epochs + 1):
            n_errors = 0
            for xi, yi in zip(X, y):
                y_hat  = self.predict(xi.reshape(1, -1))[0]
                error  = yi - y_hat          # +1, -1, or 0
                # Update weights and bias only when there is a mistake
                self.weights += self.lr * error * xi
                self.bias    += self.lr * error
                if error != 0:
                    n_errors += 1
            self.errors_per_epoch.append(n_errors)
            print(f"  Epoch {epoch:>2}/{epochs}  |  Misclassifications: {n_errors}")
            # Converged
            if n_errors == 0:
                print(f"  Converged at epoch {epoch}")
                break
        return self

    def accuracy(self, X, y):
        preds = self.predict(X)
        return float(np.mean(preds == y))

---
## 5. Train on AND — Before Training Predictions

In [ ]:
np.random.seed(42)
p_and = Perceptron(n_features=2, learning_rate=LEARNING_RATE)

print("=== AND Gate --- Predictions BEFORE Training ===")
preds_before = p_and.predict(X_and)
for x, true, pred in zip(X_and, y_and, preds_before):
    status = "CORRECT" if pred == true else "WRONG"
    print(f"  Input {x}  True: {int(true)}  Predicted: {int(pred)}  [{status}]")
print(f"  Accuracy before training: {p_and.accuracy(X_and, y_and) * 100:.1f}%")

---
## 6. Training

In [ ]:
print("=== AND Gate --- Training ===")
p_and.fit(X_and, y_and, epochs=EPOCHS)

---
## 7. After Training Predictions and Results

In [ ]:
print("=== AND Gate --- Predictions AFTER Training ===")
preds_after = p_and.predict(X_and)
for x, true, pred in zip(X_and, y_and, preds_after):
    status = "CORRECT" if pred == true else "WRONG"
    print(f"  Input {x}  True: {int(true)}  Predicted: {int(pred)}  [{status}]")
final_acc = p_and.accuracy(X_and, y_and)
print(f"\n  Final Training Accuracy: {final_acc * 100:.1f}%")
print(f"  Learned weights : {p_and.weights}")
print(f"  Learned bias    : {p_and.bias:.4f}")

---
## 8. Training Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(p_and.errors_per_epoch) + 1),
        p_and.errors_per_epoch, marker="o", color="steelblue", linewidth=2)
ax.set_title("AND Perceptron --- Misclassifications per Epoch", fontsize=14)
ax.set_xlabel("Epoch")
ax.set_ylabel("Number of Errors")
ax.set_xticks(range(1, len(p_and.errors_per_epoch) + 1))
ax.set_ylim(-0.2, max(p_and.errors_per_epoch) + 0.5)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## 9. Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(perceptron, X, y, title):
    """Plot training points and the learned decision boundary."""
    fig, ax = plt.subplots(figsize=(6, 5))

    colors = ["tomato" if label == 0 else "steelblue" for label in y]
    ax.scatter(X[:, 0], X[:, 1], c=colors, s=200, edgecolors="k", zorder=5)

    for xi, yi_true in zip(X, y):
        ax.annotate(f"({int(xi[0])},{int(xi[1])})={int(yi_true)}",
                    xy=xi, textcoords="offset points",
                    xytext=(8, 5), fontsize=10)

    w = perceptron.weights
    b = perceptron.bias
    if abs(w[1]) > 1e-8:
        x1_vals = np.linspace(-0.5, 1.5, 200)
        x2_vals = -(w[0] * x1_vals + b) / w[1]
        ax.plot(x1_vals, x2_vals, "k--", linewidth=2, label="Decision boundary")

    red_patch  = mpatches.Patch(color="tomato",    label="Class 0")
    blue_patch = mpatches.Patch(color="steelblue", label="Class 1")
    ax.legend(handles=[red_patch, blue_patch], loc="upper left")
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(title, fontsize=14)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

plot_decision_boundary(p_and, X_and, y_and, "AND Gate --- Learned Decision Boundary")

---
## 10. Training on OR Gate

In [ ]:
np.random.seed(42)
p_or = Perceptron(n_features=2, learning_rate=LEARNING_RATE)
print("=== OR Gate --- Training ===")
p_or.fit(X_or, y_or, epochs=EPOCHS)
print(f"\nFinal Accuracy: {p_or.accuracy(X_or, y_or) * 100:.1f}%")
plot_decision_boundary(p_or, X_or, y_or, "OR Gate --- Learned Decision Boundary")

---
## 11. Why a Single-Layer Perceptron Cannot Learn XOR

XOR (exclusive OR) outputs `1` only when the two inputs *differ*:

| x1 | x2 | XOR |
|----|----|----- |
| 0  | 0  |  0  |
| 0  | 1  |  1  |
| 1  | 0  |  1  |
| 1  | 1  |  0  |

If you try to draw a **single straight line** to separate the `1`s from the `0`s, it is impossible. The points labelled `1` sit at opposite corners of the unit square. This is called **linear inseparability**.

A single perceptron can only represent a **linear decision boundary**, so it will never converge on XOR no matter how many epochs you train it.

In [ ]:
np.random.seed(42)
p_xor = Perceptron(n_features=2, learning_rate=LEARNING_RATE)

print("=== XOR Gate --- Training (will NOT converge) ===")
p_xor.fit(X_xor, y_xor, epochs=100)   # 100 epochs - still won't converge

print("\n=== XOR Gate --- Predictions After 100 Epochs ===")
preds_xor = p_xor.predict(X_xor)
for x, true, pred in zip(X_xor, y_xor, preds_xor):
    status = "CORRECT" if pred == true else "WRONG"
    print(f"  Input {x}  True: {int(true)}  Predicted: {int(pred)}  [{status}]")
print(f"\n  Final Accuracy: {p_xor.accuracy(X_xor, y_xor) * 100:.1f}%")
print("  The perceptron oscillates and never achieves 100% accuracy on XOR.")

In [ ]:
# Plot XOR -- no straight line can separate the classes
fig, ax = plt.subplots(figsize=(5, 5))
colors = ["tomato" if lbl == 0 else "steelblue" for lbl in y_xor]
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=300, edgecolors="k", zorder=5)
for xi, yi in zip(X_xor, y_xor):
    ax.annotate(f"XOR={int(yi)}", xy=xi, xytext=(8, 5),
                textcoords="offset points", fontsize=11)
red_patch  = mpatches.Patch(color="tomato",    label="Class 0")
blue_patch = mpatches.Patch(color="steelblue", label="Class 1")
ax.legend(handles=[red_patch, blue_patch])
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("XOR --- No Single Linear Boundary Separates the Classes", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

---
## 12. Observations and Conclusion

- The single-layer perceptron **converges quickly** on AND and OR because both are linearly separable.
- On **XOR**, the perceptron oscillates indefinitely. Its error count never reaches zero, because no linear boundary can separate the XOR classes.
- To solve XOR, we need at least **one hidden layer** — the foundation of multi-layer perceptrons (MLPs).

| Concept | Detail |
|---|---|
| Activation | Binary step — outputs exactly 0 or 1 |
| Learning rule | Adjusts weights only on misclassified samples |
| Convergence guarantee | Guaranteed only for linearly separable datasets |
| XOR limitation | Requires a non-linear or multi-layer architecture |